# bc_daggar

In [2]:
import numpy as np
import pickle

with open("/Users/dc/cs224r/hw1/cs224r/policies/experts/Ant.pkl", "rb") as f:
    obj = pickle.load(f)

print('loading Ant.pkl')
print(f"obj, {type(obj)}")
print(obj.keys())

policy_obj = obj["GaussianPolicy"]

# this poolicy is a MPL representing a gaussian, defined with mean and std dev. or variance
policy = policy_obj  # shortcut

# ---------- Extract weights ----------
W0 = policy["hidden"]["FeedforwardNet"]["layer_0"]["AffineLayer"]["W"]
b0 = policy["hidden"]["FeedforwardNet"]["layer_0"]["AffineLayer"]["b"]

W1 = policy["hidden"]["FeedforwardNet"]["layer_2"]["AffineLayer"]["W"]
b1 = policy["hidden"]["FeedforwardNet"]["layer_2"]["AffineLayer"]["b"]

W2 = policy["out"]["AffineLayer"]["W"]
b2 = policy["out"]["AffineLayer"]["b"]

log_std = policy["logstdevs_1_Da"]

# ---------- Extract normalization ----------
stdizer = policy["obsnorm"]["Standardizer"]

mean = stdizer["mean_1_D"]
meansq = stdizer["meansq_1_D"]

var = meansq - mean**2
std = np.sqrt(np.maximum(var, 1e-8))

# ---------- Remove batch dim ----------
mean = mean.squeeze(0)
std = std.squeeze(0)
log_std = log_std.squeeze(0)

# ---------- Expert policy ----------
class AntExpertPolicy:

    def act(self, obs, deterministic=True):

        obs = np.asarray(obs, dtype=np.float32)

        if obs.ndim == 1:
            obs = obs[None, :]

        # normalize
        x = (obs - mean) / (std + 1e-8)

        # layer 1
        x = np.tanh(x @ W0 + b0)

        # layer 2
        x = np.tanh(x @ W1 + b1)

        # output mean
        mu = x @ W2 + b2

        if deterministic:
            a = mu
        else:
            a = mu + np.exp(log_std) * np.random.randn(*mu.shape)

        return a[0]

expert = AntExpertPolicy()

print("Expert reconstructed ✔")

loading Ant.pkl
obj, <class 'dict'>
dict_keys(['GaussianPolicy', 'nonlin_type'])
Expert reconstructed ✔


/var/folders/j2/z3bgs73s7_d7h21sw46sk_4c0000gn/T/ipykernel_95019/3654406774.py:5: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f)


In [14]:
import gymnasium as gym

env = gym.make("Ant-v4", use_contact_forces=True)
obs, _ = env.reset(seed=0)
print("obs dim:", obs.shape)   # should be (111,) was 27 wo contact forces
env.close()

obs dim: (111,)


In [15]:
import gymnasium as gym

env = gym.make("Ant-v4", use_contact_forces=True)
obs, _ = env.reset(seed=0)

G = 0.0
for t in range(1000):
    act = expert.act(obs, deterministic=True)
    obs, r, terminated, truncated, _ = env.step(act)
    G += float(r)
    if terminated or truncated:
        break

env.close()
print("Expert return:", G, "len:", t+1)

Expert return: 4552.665204649944 len: 1000


In [17]:
%pip install moviepy

  Using cached proglog-0.1.12-py3-none-any.whl.metadata (794 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 14.6 MB/s  0:00:00 20.8 MB/s eta 0:00:01
Using cached proglog-0.1.12-py3-none-any.whl (6.3 kB)
  Attempting uninstall: pillow
    Found existing installation: pillow 12.1.0
    Uninstalling pillow-12.1.0:
      Successfully uninstalled pillow-12.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [moviepy]━━━ 2/4 [pillow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imitation 1.0.1 requires stable-baselines3~=2.2.1, but you have stable-baselines3 2.3.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

os.makedirs("videos_expert", exist_ok=True)

env = gym.make("Ant-v4", render_mode="rgb_array", use_contact_forces=True)
env = RecordVideo(env, video_folder="videos_expert", name_prefix="expert", episode_trigger=lambda ep: True)

obs, _ = env.reset(seed=0)
G = 0.0
for t in range(1000):
    act = expert.act(obs, deterministic=True)
    obs, r, terminated, truncated, _ = env.step(act)
    G += float(r)
    if terminated or truncated:
        break

env.close()
print("Expert return:", G)
print("Saved to videos_expert/")

MoviePy - Building video /Users/dc/cs224r/hw1/videos_expert/expert-episode-0.mp4.
MoviePy - Writing video /Users/dc/cs224r/hw1/videos_expert/expert-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /Users/dc/cs224r/hw1/videos_expert/expert-episode-0.mp4
Expert return: 4552.665204649944
Saved to videos_expert/


In [5]:


with open("cs224r/expert_data/expert_data_Ant-v4.pkl", "rb") as f:
    obj = pickle.load(f)
print(type(obj), len(obj)) #list
print(type(obj[0]), type(obj[1]))


print(obj[0]['observation'].shape)
print(obj[1]['observation'].shape)
print(obj[0]['image_obs'].shape)
print(obj[1]['image_obs'].shape)
print(obj[0]['reward'].shape)
print(obj[1]['reward'].shape)
print(obj[0]['action'].shape)
print(obj[1]['action'].shape)
print(obj[0]['next_observation'].shape)
print(obj[1]['next_observation'].shape)
print(obj[0]['terminal'].shape)
print(obj[1]['terminal'].shape)

#111!! 
obs = obj[0]['observation']
t = 0                            # timestep index
o = obs[t]                       # (111,)

qpos = o[0:15]
qvel = o[15:29]
cfrc = o[29:]


print(f"qpos:{qpos.shape}")
print(f"qvel:{qvel.shape}")

print(f"qpos:{qpos}")
print(f"qvel:{qvel}")
print(f"cfrc:{cfrc}")
# verify the contact forces are nonzero for time steps

for t in [0, 1, 2, 5, 10, 50]:
    o = obs[t]
    cfrc = o[29:]
    print()
    print(t, "cfrc max abs:", float(np.max(np.abs(cfrc))))

<class 'list'> 2
<class 'dict'> <class 'dict'>
(1000, 111)
(1000, 111)
(0,)
(0,)
(1000,)
(1000,)
(1000, 8)
(1000, 8)
(1000, 111)
(1000, 111)
(1000,)
(1000,)
qpos:(15,)
qvel:(14,)
qpos:[ 0.68822366  0.9896817   0.05754574 -0.09463614 -0.09089896  0.00303502
  0.03770873 -0.00807438  0.08040225  0.05505349 -0.06585521 -0.09424642
  0.07790288 -0.08203229 -0.2480246 ]
qvel:[ 0.13172872  0.00352181  0.03103334  0.12186632 -0.1217489   0.05157095
  0.08003208  0.0782658  -0.05686773  0.04898198  0.08144254 -0.16519953
  0.          0.        ]
cfrc:[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

0 cfrc max abs: 0.0

1 cfrc max abs: 0.0

2 cfrc max abs: 0.0

5 cfrc max abs: 0.0

10 cfrc max abs: 0.0

50 cfrc max abs: 1.0


/var/folders/j2/z3bgs73s7_d7h21sw46sk_4c0000gn/T/ipykernel_71960/1829143095.py:2: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f)


In [20]:
import pickle
import numpy as np
from imitation.data.types import Transitions

DATA_PATH = "cs224r/expert_data/expert_data_Ant-v4.pkl"

with open(DATA_PATH, "rb") as f:
    trajs = pickle.load(f)

assert isinstance(trajs, list) and len(trajs) > 0
print("num trajs:", len(trajs))
print("keys:", trajs[0].keys())


obs      = np.concatenate([np.asarray(t["observation"], dtype=np.float32)      for t in trajs], axis=0)
acts     = np.concatenate([np.asarray(t["action"], dtype=np.float32)           for t in trajs], axis=0)
next_obs = np.concatenate([np.asarray(t["next_observation"], dtype=np.float32) for t in trajs], axis=0)
dones    = np.concatenate([np.asarray(t["terminal"]).astype(bool)              for t in trajs], axis=0)

infos = np.array([{} for _ in range(len(obs))], dtype=object)

transitions = Transitions(obs=obs, acts=acts, next_obs=next_obs, dones=dones, infos=infos)

print("obs:", transitions.obs.shape, transitions.obs.dtype)
print("acts:", transitions.acts.shape, transitions.acts.dtype)
print("next_obs:", transitions.next_obs.shape, transitions.next_obs.dtype)
print("dones:", transitions.dones.shape, transitions.dones.dtype)
print("infos len:", len(transitions.infos), "example:", transitions.infos[0])

num trajs: 2
keys: dict_keys(['observation', 'image_obs', 'reward', 'action', 'next_observation', 'terminal'])
obs: (2000, 111) float32
acts: (2000, 8) float32
next_obs: (2000, 111) float32
dones: (2000,) bool
infos len: 2000 example: {}


/var/folders/j2/z3bgs73s7_d7h21sw46sk_4c0000gn/T/ipykernel_77162/2910023150.py:8: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  trajs = pickle.load(f)


In [21]:
import gymnasium as gym

ENV_ID = "Ant-v4"

env = gym.make(ENV_ID, use_contact_forces=True)
o, _ = env.reset(seed=0)
print("env obs dim:", o.shape, "demo obs dim:", transitions.obs.shape[1])
assert o.shape[0] == transitions.obs.shape[1]
env.close()

env obs dim: (111,) demo obs dim: 111


In [23]:
import gymnasium as gym
import numpy as np
import torch as th

from imitation.algorithms.bc import BC
from stable_baselines3.common.policies import ActorCriticPolicy

SEED = 0
ENV_ID = "Ant-v4"

env = gym.make(ENV_ID, use_contact_forces=True)

# Create an SB3 policy with the architecture you want
policy = ActorCriticPolicy(
    observation_space=env.observation_space,
    action_space=env.action_space,
    lr_schedule=lambda _: 3e-4,     # unused by BC training but required by SB3 policy ctor
    net_arch=dict(pi=[128, 128], vf=[128, 128]),
    activation_fn=th.nn.Tanh,
)

# BC takes the policy object directly (compatible with older imitation)
bc = BC(
    observation_space=env.observation_space,
    action_space=env.action_space,
    demonstrations=transitions,
    rng=np.random.default_rng(SEED),
    policy=policy,
)

bc.train(n_epochs=50)
policy = bc.policy

env.close()
print("BC training done.")

0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.0114  |
|    entropy        | 11.4     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 479      |
|    loss           | 7.78     |
|    neglogp        | 7.79     |
|    prob_true_act  | 0.000421 |
|    samples_so_far | 32       |
--------------------------------


/opt/homebrew/anaconda3/envs/il-mujoco/lib/python3.11/site-packages/imitation/algorithms/bc.py:236: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  self._logger.record(f"bc/{k}", float(v) if v is not None else None)
34batch [00:00, 334.98batch/s]
107batch [00:00, 353.61batch/s][A
182batch [00:00, 365.65batch/s]
219batch [00:00, 364.28batch/s]
293batch [00:00, 363.73batch/s]
367batch [00:01, 363.35batch/s]
404batch [00:01, 359.42batch/s]
481batch [00:01, 371.15batch/s]
Epoch 7 of 50                  

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 500      |
|    ent_loss       | -0.00732 |
|    entropy        | 7.32     |
|    epoch          | 8        |
|    l2_loss        | 0        |
|    l2_norm        | 512      |
|    loss           | 3.36     |
|    neglogp        | 3.37     |
|    prob_true_act  | 0.0345   |
|    samples_so_far | 16032    |
--------------------------------


556batch [00:01, 365.10batch/s]
593batch [00:01, 362.06batch/s]
668batch [00:01, 365.58batch/s]
706batch [00:01, 367.43batch/s]
783batch [00:02, 372.06batch/s]
858batch [00:02, 366.46batch/s]
895batch [00:02, 360.11batch/s]
971batch [00:02, 366.79batch/s]
Epoch 15 of 50                 

...

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 2500     |
|    ent_loss       | 0.00815  |
|    entropy        | -8.15    |
|    epoch          | 40       |
|    l2_loss        | 0        |
|    l2_norm        | 614      |
|    loss           | -11.2    |
|    neglogp        | -11.2    |
|    prob_true_act  | 8.48e+04 |
|    samples_so_far | 80032    |
--------------------------------


2532batch [00:06, 377.91batch/s]
2571batch [00:06, 379.65batch/s]
2649batch [00:07, 382.83batch/s]
2727batch [00:07, 385.41batch/s]
2767batch [00:07, 387.31batch/s]
2845batch [00:07, 381.57batch/s]
2884batch [00:07, 375.07batch/s]
2960batch [00:07, 374.30batch/s]
2998batch [00:08, 369.39batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 3000     |
|    ent_loss       | 0.0114   |
|    entropy        | -11.4    |
|    epoch          | 48       |
|    l2_loss        | 0        |
|    l2_norm        | 637      |
|    loss           | -13.7    |
|    neglogp        | -13.7    |
|    prob_true_act  | 1.39e+06 |
|    samples_so_far | 96032    |
--------------------------------


3035batch [00:08, 368.83batch/s]
3074batch [00:08, 372.78batch/s]
3100batch [00:08, 370.19batch/s]

BC training done.


In [24]:
import numpy as np
import gymnasium as gym

def eval_policy(policy, env_id="Ant-v4", n_episodes=10, seed=0, max_steps=1000):
    env = gym.make(env_id, use_contact_forces=True)
    rets, lens = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        G = 0.0
        t = 0
        while t < max_steps:
            act, _ = policy.predict(obs, deterministic=True)
            obs, r, terminated, truncated, _ = env.step(act)
            G += float(r)
            t += 1
            if terminated or truncated:
                break
        rets.append(G)
        lens.append(t)
    env.close()
    return np.array(rets), np.array(lens)

rets, lens = eval_policy(policy, n_episodes=10, seed=SEED)
print("BC return mean/std:", float(rets.mean()), float(rets.std()))
print("Returns:", rets)

BC return mean/std: 4462.9217717666415 86.17920360363793
Returns: [4496.94048186 4345.71321194 4400.07496032 4542.34236782 4547.10937042
 4609.87519328 4412.45965977 4428.47258351 4505.36724288 4340.86264586]


In [25]:
import torch

SAVE_PATH = "bc_ant_contact111_state_dict.pt"
torch.save(policy.state_dict(), SAVE_PATH)
print("Saved:", SAVE_PATH)

# Reload into same policy class
env = gym.make(ENV_ID, use_contact_forces=True)
policy2 = ActorCriticPolicy(
    observation_space=env.observation_space,
    action_space=env.action_space,
    lr_schedule=lambda _: 3e-4,
    net_arch=dict(pi=[128, 128], vf=[128, 128]),
    activation_fn=th.nn.Tanh,
)
policy2.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
env.close()

rets2, _ = eval_policy(policy2, n_episodes=5, seed=123)
print("Reloaded mean:", float(rets2.mean()))

Saved: bc_ant_contact111_state_dict.pt
Reloaded mean: 4321.123908174828


In [26]:
import os
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

os.makedirs("videos_bc", exist_ok=True)

env = None
try:
    env = gym.make("Ant-v4", render_mode="rgb_array", use_contact_forces=True)
    env = RecordVideo(env, video_folder="videos_bc", name_prefix="bc", episode_trigger=lambda ep: True)

    obs, _ = env.reset(seed=0)
    G = 0.0
    for t in range(1000):
        act, _ = policy.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(act)
        G += float(r)
        if terminated or truncated:
            break

    print("BC return:", G)
finally:
    if env is not None:
        env.close()

print("Saved MP4(s) in videos_bc/")

/opt/homebrew/anaconda3/envs/il-mujoco/lib/python3.11/site-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /Users/dc/cs224r/hw1/videos_bc folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video /Users/dc/cs224r/hw1/videos_bc/bc-episode-0.mp4.
MoviePy - Writing video /Users/dc/cs224r/hw1/videos_bc/bc-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /Users/dc/cs224r/hw1/videos_bc/bc-episode-0.mp4
BC return: 4496.940481864391
Saved MP4(s) in videos_bc/


In [ ]:
BC return ≈ 4497 (close to your expert ≈ 4553), which is very plausible given you only had 2 demos and the eval is on the same observation definition (use_contact_forces=True).

In [27]:
import os, gymnasium as gym
from gymnasium.wrappers import RecordVideo

os.makedirs("videos_compare", exist_ok=True)

def record(policy_like, name):
    env = None
    try:
        env = gym.make("Ant-v4", render_mode="rgb_array", use_contact_forces=True)
        env = RecordVideo(env, "videos_compare", name_prefix=name, episode_trigger=lambda ep: True)
        obs, _ = env.reset(seed=0)
        G = 0.0
        for _ in range(1000):
            if hasattr(policy_like, "predict"):      # SB3 policy
                act, _ = policy_like.predict(obs, deterministic=True)
            else:                                   # your numpy expert
                act = policy_like.act(obs, deterministic=True)
            obs, r, terminated, truncated, _ = env.step(act)
            G += float(r)
            if terminated or truncated:
                break
        print(name, "return:", G)
    finally:
        if env is not None:
            env.close()

record(expert, "expert")
record(policy, "bc")
print("Wrote videos in videos_compare/")

/opt/homebrew/anaconda3/envs/il-mujoco/lib/python3.11/site-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /Users/dc/cs224r/hw1/videos_compare folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video /Users/dc/cs224r/hw1/videos_compare/expert-episode-0.mp4.
MoviePy - Writing video /Users/dc/cs224r/hw1/videos_compare/expert-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /Users/dc/cs224r/hw1/videos_compare/expert-episode-0.mp4
expert return: 4552.665204649944
MoviePy - Building video /Users/dc/cs224r/hw1/videos_compare/bc-episode-0.mp4.
MoviePy - Writing video /Users/dc/cs224r/hw1/videos_compare/bc-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /Users/dc/cs224r/hw1/videos_compare/bc-episode-0.mp4
bc return: 4496.940481864391
Wrote videos in videos_compare/


In [28]:
import os
import numpy as np
import gymnasium as gym
import torch as th
from imitation.data.types import Transitions
from imitation.algorithms.bc import BC
from stable_baselines3.common.policies import ActorCriticPolicy


# -----------------------------
# Assumes you already have:
#   - expert: object with expert.act(obs, deterministic=True) -> action (8,)
#   - trajs: list of trajectory dicts loaded from expert_data_Ant-v4.pkl
# or you can start with empty dataset (pure DAgger) if you want.
# -----------------------------

ENV_ID = "Ant-v4"
USE_CONTACT_FORCES = True
SEED = 0

# DAgger knobs
DAGGER_ITERS = 10            # number of DAgger iterations
ROLLOUT_EPISODES_PER_ITER = 2
MAX_STEPS = 1000
BC_EPOCHS_PER_ITER = 20      # train student on aggregated data each iter
NET = dict(pi=[128, 128], vf=[128, 128])  # student arch


def make_infos(n):
    infos = np.empty(n, dtype=object)
    infos[:] = {}
    return infos


def make_transitions(obs, acts, next_obs, dones):
    return Transitions(
        obs=np.asarray(obs, dtype=np.float32),
        acts=np.asarray(acts, dtype=np.float32),
        next_obs=np.asarray(next_obs, dtype=np.float32),
        dones=np.asarray(dones, dtype=bool),
        infos=make_infos(len(obs)),
    )


def flatten_demo_trajs(trajs):
    obs      = np.concatenate([np.asarray(t["observation"], dtype=np.float32)      for t in trajs], axis=0)
    acts     = np.concatenate([np.asarray(t["action"], dtype=np.float32)           for t in trajs], axis=0)
    next_obs = np.concatenate([np.asarray(t["next_observation"], dtype=np.float32) for t in trajs], axis=0)
    dones    = np.concatenate([np.asarray(t["terminal"]).astype(bool)              for t in trajs], axis=0)
    return obs, acts, next_obs, dones


def collect_dagger_rollouts(student_policy, expert_policy, n_episodes, seed):
    """
    Roll out student; label each visited state with expert action.
    Returns arrays: obs, expert_acts, next_obs, dones and a list of episode returns.
    """
    env = gym.make(ENV_ID, use_contact_forces=USE_CONTACT_FORCES)

    all_obs, all_exp_acts, all_next_obs, all_dones = [], [], [], []
    ep_returns = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        G = 0.0

        for t in range(MAX_STEPS):
            # student acts in env
            act_student, _ = student_policy.predict(obs, deterministic=True)

            # expert label for *current* obs
            act_expert = expert_policy.act(obs, deterministic=True)

            next_obs, r, terminated, truncated, _ = env.step(act_student)
            done = bool(terminated or truncated)

            all_obs.append(obs)
            all_exp_acts.append(act_expert)
            all_next_obs.append(next_obs)
            all_dones.append(done)

            G += float(r)
            obs = next_obs

            if done:
                break

        ep_returns.append(G)

    env.close()
    return (
        np.asarray(all_obs, dtype=np.float32),
        np.asarray(all_exp_acts, dtype=np.float32),
        np.asarray(all_next_obs, dtype=np.float32),
        np.asarray(all_dones, dtype=bool),
        ep_returns,
    )


def eval_student(student_policy, n_episodes=5, seed=123):
    env = gym.make(ENV_ID, use_contact_forces=USE_CONTACT_FORCES)
    rets = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        G = 0.0
        for _ in range(MAX_STEPS):
            a, _ = student_policy.predict(obs, deterministic=True)
            obs, r, terminated, truncated, _ = env.step(a)
            G += float(r)
            if terminated or truncated:
                break
        rets.append(G)
    env.close()
    return float(np.mean(rets)), float(np.std(rets))


# -----------------------------
# 0) Build initial dataset (recommended: start from your demos)
# -----------------------------
# If you already have `trajs` loaded (2 expert trajectories), use them as seed data:
#   obs0, acts0, next_obs0, dones0 = flatten_demo_trajs(trajs)
#   acts0 are expert actions already.
#
# Or start from nothing (pure DAgger): set arrays empty.
#
USE_SEED_DEMOS = True

if USE_SEED_DEMOS:
    obs0, acts0, next_obs0, dones0 = flatten_demo_trajs(trajs)
else:
    obs0      = np.zeros((0, 111), dtype=np.float32)
    acts0     = np.zeros((0, 8), dtype=np.float32)
    next_obs0 = np.zeros((0, 111), dtype=np.float32)
    dones0    = np.zeros((0,), dtype=bool)

agg_obs, agg_acts, agg_next_obs, agg_dones = obs0, acts0, next_obs0, dones0
transitions = make_transitions(agg_obs, agg_acts, agg_next_obs, agg_dones)

print("Initial dataset size:", len(transitions.obs))


# -----------------------------
# 1) Create student policy + BC trainer
# -----------------------------
env = gym.make(ENV_ID, use_contact_forces=USE_CONTACT_FORCES)

student_policy = ActorCriticPolicy(
    observation_space=env.observation_space,
    action_space=env.action_space,
    lr_schedule=lambda _: 3e-4,   # required by ctor; BC training uses supervised loss
    net_arch=NET,
    activation_fn=th.nn.Tanh,
)

bc = BC(
    observation_space=env.observation_space,
    action_space=env.action_space,
    demonstrations=transitions,
    rng=np.random.default_rng(SEED),
    policy=student_policy,
)

env.close()


# -----------------------------
# 2) DAgger loop
# -----------------------------
log = []
for it in range(DAGGER_ITERS):
    # (a) Train student on aggregated dataset
    bc.set_demonstrations(transitions) if hasattr(bc, "set_demonstrations") else None
    bc.train(n_epochs=BC_EPOCHS_PER_ITER)

    # (b) Collect rollouts with student; label with expert actions
    obs_i, exp_acts_i, next_obs_i, dones_i, student_rollout_returns = collect_dagger_rollouts(
        student_policy=bc.policy,
        expert_policy=expert,
        n_episodes=ROLLOUT_EPISODES_PER_ITER,
        seed=SEED + 1000 * it,
    )

    # (c) Aggregate
    agg_obs      = np.concatenate([agg_obs, obs_i], axis=0)
    agg_acts     = np.concatenate([agg_acts, exp_acts_i], axis=0)
    agg_next_obs = np.concatenate([agg_next_obs, next_obs_i], axis=0)
    agg_dones    = np.concatenate([agg_dones, dones_i], axis=0)

    transitions = make_transitions(agg_obs, agg_acts, agg_next_obs, agg_dones)

    # (d) Evaluate student
    mean_ret, std_ret = eval_student(bc.policy, n_episodes=5, seed=SEED + 10 * it)

    row = dict(
        iter=it,
        dataset_size=len(agg_obs),
        rollout_return_mean=float(np.mean(student_rollout_returns)),
        rollout_return_std=float(np.std(student_rollout_returns)),
        eval_return_mean=mean_ret,
        eval_return_std=std_ret,
    )
    log.append(row)
    print(row)

# final policy
dagger_policy = bc.policy
print("DAgger done. Final dataset size:", len(agg_obs))

Initial dataset size: 2000


0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.0114  |
|    entropy        | 11.4     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 479      |
|    loss           | 7.7      |
|    neglogp        | 7.72     |
|    prob_true_act  | 0.00045  |
|    samples_so_far | 32       |
--------------------------------


49batch [00:00, 246.41batch/s]
102batch [00:00, 259.07batch/s][A
180batch [00:00, 245.09batch/s]
234batch [00:00, 258.23batch/s]
289batch [00:01, 264.39batch/s]
370batch [00:01, 261.80batch/s]
425batch [00:01, 265.84batch/s]
479batch [00:01, 256.23batch/s]
Epoch 7 of 20                  

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 500      |
|    ent_loss       | -0.00732 |
|    entropy        | 7.32     |
|    epoch          | 8        |
|    l2_loss        | 0        |
|    l2_norm        | 513      |
|    loss           | 3.36     |
|    neglogp        | 3.37     |
|    prob_true_act  | 0.0344   |
|    samples_so_far | 16032    |
--------------------------------


...

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 11500    |
|    ent_loss       | 0.0212   |
|    entropy        | -21.2    |
|    epoch          | 18       |
|    l2_loss        | 0        |
|    l2_norm        | 926      |
|    loss           | -21      |
|    neglogp        | -21      |
|    prob_true_act  | 1.64e+10 |
|    samples_so_far | 368032   |
--------------------------------


11843batch [00:37, 335.42batch/s]
11975batch [00:38, 316.29batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 12000    |
|    ent_loss       | 0.0212   |
|    entropy        | -21.2    |
|    epoch          | 19       |
|    l2_loss        | 0        |
|    l2_norm        | 928      |
|    loss           | -21.1    |
|    neglogp        | -21.2    |
|    prob_true_act  | 7.79e+09 |
|    samples_so_far | 384032   |
--------------------------------


12470batch [00:39, 323.17batch/s]
12500batch [00:39, 313.73batch/s]


{'iter': 9, 'dataset_size': 22000, 'rollout_return_mean': 4714.772917093098, 'rollout_return_std': 3.9157624505710373, 'eval_return_mean': 4683.164497808877, 'eval_return_std': 122.39245456568634}
DAgger done. Final dataset size: 22000


In [29]:
import torch

torch.save(dagger_policy.state_dict(), "dagger_ant_contact111_state_dict.pt")
print("Saved dagger policy weights.")

Saved dagger policy weights.


In [30]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

os.makedirs("videos_dagger", exist_ok=True)

env = None
try:
    env = gym.make("Ant-v4", render_mode="rgb_array", use_contact_forces=True)
    env = RecordVideo(env, video_folder="videos_dagger", name_prefix="dagger", episode_trigger=lambda ep: True)

    obs, _ = env.reset(seed=0)
    G = 0.0
    for _ in range(1000):
        act, _ = dagger_policy.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(act)
        G += float(r)
        if terminated or truncated:
            break

    print("DAgger return:", G)
finally:
    if env is not None:
        env.close()

print("Saved MP4(s) in videos_dagger/")

/opt/homebrew/anaconda3/envs/il-mujoco/lib/python3.11/site-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /Users/dc/cs224r/hw1/videos_dagger folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video /Users/dc/cs224r/hw1/videos_dagger/dagger-episode-0.mp4.
MoviePy - Writing video /Users/dc/cs224r/hw1/videos_dagger/dagger-episode-0.mp4



MoviePy - Done !
MoviePy - video ready /Users/dc/cs224r/hw1/videos_dagger/dagger-episode-0.mp4
DAgger return: 4940.300045259495
Saved MP4(s) in videos_dagger/
